# AirCommand — YOLO gesture detector training (Google Colab)

Fine-tunes **YOLO11n** (default) or **YOLOv8n** on a subject-disjoint subset of **HaGRIDv2**
for the AirCommand gesture vocabulary: `one, fist, palm, like, peace, no_gesture`.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better).

What this notebook does:
1. Gets the AirCommand code (GitHub clone or ZIP upload).
2. Builds a YOLO dataset by streaming only the sampled images out of the 119 GB HaGRIDv2-512px
   archive via HTTP Range requests (no full download). Splits are HaGRID's official subject
   splits; `dataset_setup.py` re-verifies there is no subject overlap.
3. Fine-tunes from COCO-pretrained weights, checkpointing to Google Drive.
4. Evaluates on **val** and on the held-out **test** split: per-class precision/recall,
   mAP50, mAP50-95, confusion matrix, GPU and CPU inference speed.
5. Exports `best.pt` for local use (`models/best.pt`).

Dataset license: HaGRIDv2 is released under a variant of CC BY-SA 4.0
(see https://github.com/hukenovs/hagrid/tree/master/license). Respect its terms.

Nothing in this notebook has pre-filled results; every number comes from your run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to GPU first."
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Get the AirCommand code
Set `REPO_URL` to your GitHub repository, **or** leave it empty and upload `AirCommand.zip`
when prompted.

In [ ]:
import os, subprocess, zipfile, pathlib, shutil

REPO_URL = ""   # e.g. "https://github.com/<you>/AirCommand.git"; empty -> upload AirCommand.zip
WORK = pathlib.Path("/content")
os.chdir(WORK)

def find_repo(base: pathlib.Path):
    for p in [base, *base.glob("*"), *base.glob("*/*")]:
        if (p / "training" / "dataset_setup.py").is_file():
            return p
    return None

REPO = find_repo(WORK / "AirCommand")
if REPO is None:
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK / "AirCommand")], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()  # choose AirCommand.zip
        zip_name = next(n for n in uploaded if n.endswith(".zip"))
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall(WORK / "AirCommand")
    REPO = find_repo(WORK / "AirCommand")
assert REPO is not None, "Could not find training/dataset_setup.py in the cloned/uploaded code"
os.chdir(REPO)
print("Repository:", REPO)

In [ ]:
# Same Ultralytics range as requirements.txt so best.pt loads locally.
%pip install -q "ultralytics>=8.3.0,<8.5.0" requests pyyaml pandas
import ultralytics
ultralytics.checks()

## 2. Settings
`MODEL_NAME = "yolov8n.pt"` switches to YOLOv8n. Per-class quotas control dataset size and
download time (defaults: 2500 / 400 / 600 images per class for train / val / test).

In [ ]:
MODEL_NAME = "yolo11n.pt"      # or "yolov8n.pt"
EPOCHS = 40
IMGSZ = 640
BATCH = 32                     # lower to 16 if you hit CUDA out-of-memory
PATIENCE = 15
TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 2500, 400, 600
WORKERS_DL = 16                # parallel image downloads

USE_DRIVE = True               # keep checkpoints + dataset cache across Colab disconnects
DATA_DIR = pathlib.Path("/content/datasets/aircommand")
RUN_NAME = MODEL_NAME.replace(".pt", "") + "_aircommand"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/AirCommand")
else:
    DRIVE_DIR = pathlib.Path("/content/aircommand_runs")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = DRIVE_DIR / "runs"
DATASET_CACHE = DRIVE_DIR / f"dataset_{TRAIN_PER_CLASS}_{VAL_PER_CLASS}_{TEST_PER_CLASS}.tar"
print("Outputs go to", DRIVE_DIR)

## 3. Build the dataset
First run: reads the archive index (~1M entries; can take a few minutes and ~1-2 GB RAM),
then downloads only the sampled images. Re-running resumes. The finished dataset is cached
as a tarball on Drive so later sessions skip the download.

If the HaGRID server refuses Range requests the script stops with instructions; in that
case download `hagridv2_512.zip` + `annotations.zip` yourself (links in the HaGRID README)
and pass `--images-zip` / `--annotations-zip`.

In [ ]:
import tarfile, json
if (DATA_DIR / "data.yaml").is_file():
    print("Dataset already present at", DATA_DIR)
elif DATASET_CACHE.is_file():
    print("Restoring dataset from", DATASET_CACHE)
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DATASET_CACHE) as tf:
        tf.extractall(DATA_DIR.parent)
else:
    cmd = ["python", "training/dataset_setup.py", "--out", str(DATA_DIR),
           "--train-per-class", str(TRAIN_PER_CLASS), "--val-per-class", str(VAL_PER_CLASS),
           "--test-per-class", str(TEST_PER_CLASS), "--workers", str(WORKERS_DL)]
    print(" ".join(cmd))
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        raise SystemExit(f"dataset_setup.py exited with {proc.returncode}; re-run this cell to resume.")
    with tarfile.open(DATASET_CACHE, "w") as tf:
        tf.add(DATA_DIR, arcname=DATA_DIR.name)
    print("Cached dataset to", DATASET_CACHE)

# data.yaml stores an absolute path; rewrite it in case the dataset was restored elsewhere.
yaml_text = (DATA_DIR / "data.yaml").read_text().splitlines()
yaml_text[0] = f"path: {DATA_DIR.as_posix()}"
(DATA_DIR / "data.yaml").write_text("\n".join(yaml_text) + "\n")
DATA_YAML = str(DATA_DIR / "data.yaml")
report = json.loads((DATA_DIR / "split_report.json").read_text())
print(json.dumps({"splits": report["splits"], "leakage": report["leakage"]}, indent=2))

In [ ]:
# Visual sanity check of converted labels: boxes must sit on hands with the right class.
import random, cv2, yaml
import matplotlib.pyplot as plt
names = yaml.safe_load(open(DATA_YAML))["names"]
imgs = sorted((DATA_DIR / "images" / "train").glob("*.jpg"))
random.Random(1).shuffle(imgs)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, imgs[:8]):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for line in (DATA_DIR / "labels" / "train" / (p.stem + ".txt")).read_text().split("\n"):
        if not line.strip():
            continue
        c, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        x1, y1 = int(cx - bw / 2), int(cy - bh / 2)
        cv2.rectangle(im, (x1, y1), (int(x1 + bw), int(y1 + bh)), (255, 0, 0), 3)
        cv2.putText(im, names[int(c)], (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)
    ax.imshow(im); ax.set_title(p.name[:28]); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Fine-tune
Loads COCO-pretrained `MODEL_NAME` and fine-tunes all layers. Checkpoints (`last.pt`, `best.pt`,
and every 5 epochs) are written to Drive. `best.pt` is selected on the **val** split only;
the **test** split stays untouched until section 6.

`fliplr=0.5` is safe because all five gestures are left/right symmetric (HaGRID contains both
hands anyway). If Colab disconnects, run the *resume* cell instead of this one.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL_NAME)  # downloads COCO-pretrained weights on first use
train_results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
    project=str(PROJECT_DIR), name=RUN_NAME, exist_ok=True,
    seed=0, deterministic=True, cos_lr=True, close_mosaic=10, fliplr=0.5,
    workers=2, plots=True, save_period=5,
)
RUN_DIR = pathlib.Path(train_results.save_dir) if train_results is not None else PROJECT_DIR / RUN_NAME
print("Run directory:", RUN_DIR)

In [ ]:
# RESUME after a disconnect (run the settings + dataset cells first, then this instead of training).
RUN_DIR = PROJECT_DIR / RUN_NAME
last = RUN_DIR / "weights" / "last.pt"
if last.is_file():
    YOLO(str(last)).train(resume=True)
else:
    print("No last.pt found at", last)

## 5. Validation split (used for model selection)

In [ ]:
import pandas as pd
from IPython.display import Image, display

RUN_DIR = PROJECT_DIR / RUN_NAME
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.is_file(), f"{BEST} not found; train first."
best_model = YOLO(str(BEST))
CLASS_NAMES = [best_model.names[i] for i in sorted(best_model.names)]
print("Classes in best.pt:", CLASS_NAMES)

def per_class_table(metrics):
    rows = []
    for i, c in enumerate(metrics.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(i)
        rows.append({"class": CLASS_NAMES[int(c)], "precision": p, "recall": r, "mAP50": ap50, "mAP50-95": ap})
    rows.append({"class": "all", "precision": metrics.box.mp, "recall": metrics.box.mr,
                 "mAP50": metrics.box.map50, "mAP50-95": metrics.box.map})
    return pd.DataFrame(rows).set_index("class").round(4)

val_metrics = best_model.val(data=DATA_YAML, split="val", imgsz=IMGSZ, batch=BATCH, plots=True,
                             project=str(RUN_DIR), name="eval_val", exist_ok=True)
val_table = per_class_table(val_metrics)
display(val_table)
print("Speed (ms/img, GPU):", val_metrics.speed)

## 6. Held-out test split
Subjects in this split never appear in train or val. Report **these** numbers in the README.

In [ ]:
import numpy as np
test_metrics = best_model.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH, plots=True,
                              project=str(RUN_DIR), name="eval_test", exist_ok=True)
test_table = per_class_table(test_metrics)
display(test_table)
print("Speed (ms/img, GPU):", test_metrics.speed)

test_dir = pathlib.Path(test_metrics.save_dir)
for png in ("confusion_matrix_normalized.png", "confusion_matrix.png", "PR_curve.png", "BoxPR_curve.png"):
    if (test_dir / png).is_file():
        print(png); display(Image(filename=str(test_dir / png), width=700))

# Top confusions from the raw matrix. Ultralytics stores matrix[predicted, true] with an extra
# final 'background' row/column (missed objects / false positives).
cm_obj = getattr(test_metrics, "confusion_matrix", None)
if cm_obj is not None and getattr(cm_obj, "matrix", None) is not None:
    m = np.asarray(cm_obj.matrix)
    labels = CLASS_NAMES + ["background"]
    pairs = [(m[p, t], labels[t], labels[p]) for p in range(m.shape[0]) for t in range(m.shape[1])
             if p != t and m[p, t] > 0]
    print("Most frequent confusions (count, true -> predicted):")
    for cnt, t, p in sorted(pairs, reverse=True)[:10]:
        print(f"  {int(cnt):5d}  {t} -> {p}")
else:
    print("Raw confusion matrix not exposed by this Ultralytics version; use the PNG above.")

## 7. CPU inference speed (closer to a laptop without a GPU)
Colab CPUs are not your machine; treat this as a rough indication and measure locally with
`python -m src.main --benchmark 300 --dry-run`.

In [ ]:
import time, statistics, cv2
test_imgs = sorted((DATA_DIR / "images" / "test").glob("*.jpg"))[:100]
cpu_model = YOLO(str(BEST))
for p in test_imgs[:5]:
    cpu_model.predict(str(p), imgsz=IMGSZ, device="cpu", verbose=False)  # warm-up
times = []
for p in test_imgs:
    img = cv2.imread(str(p))
    t0 = time.perf_counter()
    cpu_model.predict(img, imgsz=IMGSZ, device="cpu", verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
times.sort()
cpu_speed = {"images": len(times), "mean_ms": round(statistics.mean(times), 1),
             "p50_ms": round(times[len(times) // 2], 1), "p95_ms": round(times[int(0.95 * (len(times) - 1))], 1)}
print("CPU end-to-end predict():", cpu_speed)

## 8. Save metrics and export weights

In [ ]:
import platform
summary = {
    "model": MODEL_NAME, "epochs_requested": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "dataset": report["splits"], "leakage_check": report["leakage"],
    "val": val_table.reset_index().to_dict(orient="records"),
    "test": test_table.reset_index().to_dict(orient="records"),
    "gpu_speed_ms_per_img_test": test_metrics.speed,
    "cpu_predict_ms": cpu_speed,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ultralytics": ultralytics.__version__, "torch": torch.__version__, "python": platform.python_version(),
}
(RUN_DIR / "aircommand_metrics.json").write_text(json.dumps(summary, indent=2, default=float))
shutil.copy(BEST, DRIVE_DIR / "best.pt")
print("Saved", RUN_DIR / "aircommand_metrics.json", "and", DRIVE_DIR / "best.pt")

EXPORT_ONNX = False  # optional; the app uses best.pt
if EXPORT_ONNX:
    print(best_model.export(format="onnx", imgsz=IMGSZ, dynamic=False))

from google.colab import files
files.download(str(BEST))
files.download(str(RUN_DIR / "aircommand_metrics.json"))

## 9. Use the weights locally
1. Put the downloaded file at `AirCommand/models/best.pt`.
2. Keep `model.classes` in `config/gestures.yaml` identical to `CLASS_NAMES` printed above.
3. Run `python -m src.main --dry-run` first, then `python -m src.main`.
4. Copy the test-split table and the confusion matrix into the README's *Results* section.